# 01 · LoRA Anatomy — the host, the recipes, the adapters

**How do you adapt a pretrained separator to a shifted domain at ~1% of its
parameters?** This notebook is the visual companion to [`../THEORY.md`](../THEORY.md):
it tours the **mock** Open-Unmix host (shape-faithful to `umxhq`, no weight download),
shows where the five recipes attach, visualizes LoRA (the $B=0$ start and rank as a
dial), renders the exact trainable-parameters table, illustrates the two domains, and
demonstrates the merge round-trip. It sets up the experiments in
[`02_lora_adaptation_experiments.ipynb`](02_lora_adaptation_experiments.ipynb).

Nothing here needs a GPU or a weight download. Most cells run on CPU from the mock
model alone; the two that need a decoded MUSDB shard (the real 64 kbps spectrogram
overlay) are marked **⚠️ RUN THIS LATER**. The notebook ships **un-executed** so the
committed file is a clean scaffold.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §3.1 (the five recipes),
  §3.2 (the two domains), §8 (code contracts); [`../THEORY.md`](../THEORY.md) §2 (LoRA
  math), §3 (LoRA on the BiLSTM — the crux), §4 (the host + share table).
- **Data prep is *not* repeated here.** Acquisition, licensing, the 86/14/50 split,
  and the STFT front end live in Direction 01's notebooks
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)).
  This direction reuses them verbatim and consumes the **stereo** shards.
- **Code, not prose, is authoritative:** adapters are `singnet/peft/lora.py`; the host
  and recipes are `singnet/peft/umx_wrapper.py`; every number below is asserted in
  `tests/` (gate G0). No weights are downloaded — the mock host is byte-shape-identical
  to `umxhq`.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · The mock host — a shape-faithful `umxhq`

`MockOpenUnmix` mirrors the real Open-Unmix vocals model exactly (verified shapes,
00-shared-research/papers/openunmix2019.md §2): input cropped to **1487** bandwidth
bins, `fc1`(2·1487→512) → `bn1` → tanh → **BiLSTM**(512, 256, 3 layers, bidir) →
skip-concat(1024) → `fc2`(1024→512) → `bn2` → relu → `fc3`(512→2·**2049**) → `bn3` →
output affine → relu → ×mixture. The load-bearing subtlety: the mask spans the **full**
2049-bin spectrum, so `fc3` outputs 2·2049=4098, **not** 2·1487.

In [ ]:
# CPU-runnable later: build the mock host (no download) and print the shape table.
import torch
from singnet.peft import load_umxhq, count_total_params, trainable_report

model = load_umxhq('cpu', mock=True)
print('host total params:', f'{count_total_params(model):,}', '(pinned 8,893,348)')
report = trainable_report(model)
# show the big weight tensors and the learnable scale/mean vectors
print(report[report['n_params'] > 1000].to_string(index=False))

## 2 · Where the five recipes attach

`apply_recipe(model, recipe)` freezes/wraps in place (MASTER_PLAN §3.1): `zeroshot`
trains nothing; `head` trains `fc3`+`bn3`+output affine; `lora4`/`lora16` freeze all
base weights and wrap `fc1`/`fc2`/`fc3` + all 12 LSTM `weight_ih`/`weight_hh` with
LoRA, plus the input/output affine; `full` trains everything. The exact trainable
parameter-name set per recipe is unit-tested (`tests/test_umx_wrapper.py`).

In [ ]:
# CPU-runnable later: apply each recipe to a fresh mock and list its trainable set.
from singnet.peft import load_umxhq, apply_recipe, trainable_param_names, measured_trainable_share

for recipe in ('zeroshot', 'head', 'lora4', 'lora16', 'full'):
    m = load_umxhq('cpu', mock=True)
    apply_recipe(m, recipe)
    names = sorted(trainable_param_names(m))
    preview = names[:4] + (['…'] if len(names) > 4 else [])
    print(f'{recipe:9s} | {len(names):3d} trainable tensors | '
          f'{measured_trainable_share(m)*100:8.4f}% of host | {preview}')

## 3 · LoRA visualized — the $B=0$ start and rank as a dial

$h = W_0x + \frac{\alpha}{r}B(Ax)$ with $A\sim\mathcal N(0,\sigma^2)$, $B=0$, so at
init $\Delta W=0$ and the adapted map equals the frozen one **exactly** — for a
LoRA-wrapped host this start *is* zero-shot (THEORY §2.1). The rank $r$ dials the
capacity of the update at $r(d+k)$ params per matrix. The cell below shows the
bit-exact $B=0$ identity and the growth of trained params with $r$.

In [ ]:
# CPU-runnable later: B=0 identity, then rank-vs-params for one wrapped matrix.
import torch
from torch import nn
from singnet.peft import LoRALinear
from singnet.peft.lora import lora_param_cost

base = nn.Linear(512, 1024, bias=False)
x = torch.randn(8, 512)
wrapped = LoRALinear(base, r=16, alpha=32)
print('B=0 identity — max abs diff:', float((wrapped(x) - base(x)).abs().max()), '(exactly 0.0)')
print('rank r -> trainable params for a 1024x512 matrix (= r(d_out+d_in)):')
for r in (1, 2, 4, 8, 16, 32):
    print(f'  r={r:2d}: {lora_param_cost(1024, 512, r):>7,}  vs full 524,288')

## 4 · The trainable-parameters table + a log-scale chart

The exact recipe → trained-parameter accounting (host base 8,893,348; share of the
host, the H-05a denominator). `head` is the **inefficient** baseline (23.7%) the LoRA
recipes must beat; `lora4`/`lora16` sit at 1.27%/4.85% — both under the H-05a 5%
budget. These are the numbers `tests/test_umx_wrapper.py` pins (see also
`results/DEVIATIONS.md` for the head-share correction vs the plan's 18% estimate).

In [ ]:
# CPU-runnable later: the exact table + a log-scale params-vs-recipe bar chart.
import matplotlib.pyplot as plt
from singnet.peft import recipe_share_table

table = recipe_share_table()
print(table.to_string(index=False))

trained = table.set_index('recipe')['trainable_params'].clip(lower=1)  # clip zeroshot for log
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(trained.index, trained.values, color=['#bbb', '#d9963a', '#3a8fd9', '#2b6cb0', '#c0392b'])
ax.set_yscale('log'); ax.set_ylabel('trained parameters (log)')
ax.set_title('Trainable parameters per recipe (host = 8,893,348)')
for i, v in enumerate(trained.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=8)
fig.tight_layout(); plt.show()

## 5 · The two domains illustrated

**T1 (codec, primary):** every stem re-encoded to AAC 64 kbps then re-summed — a
covariate shift on **both** the input mixture and the target vocals that HQ-trained
`umxhq` never saw. Additivity is preserved by construction (THEORY §5.2). **T2
(secondary):** a genre subset if per-track labels materialize at G0b, else pink noise
at an exact 12 dB SNR added to the mixture (targets stay clean). The noise op is
CPU-runnable now; the real 64 kbps spectrogram overlay needs one decoded shard + ffmpeg.

In [ ]:
# CPU-runnable later: the T2 noise op at an EXACT 12 dB SNR (seeded, pure NumPy).
import sys; sys.path.insert(0, '../../scripts')
import numpy as np, matplotlib.pyplot as plt
import make_domains as md

sr = 44100
sig = (0.2 * np.sin(2*np.pi*220*np.arange(sr)/sr)).astype('float32')
noisy = md.add_noise_at_snr(sig, np.random.default_rng(0), snr_db=12.0)
print('achieved SNR:', round(md.measured_snr_db(sig, noisy.astype('float64') - sig), 6), 'dB (target 12.0)')
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.plot(sig[:400], label='clean mixture', lw=1)
ax.plot(noisy[:400], label='+ pink noise @ 12 dB', lw=0.8, alpha=0.8)
ax.legend(fontsize=8); ax.set_title('T2 noise op (exact SNR)'); fig.tight_layout(); plt.show()

In [ ]:
# ⚠️ RUN THIS LATER (CPU, needs one decoded stereo shard + ffmpeg) — the T1 overlay:
# a vocal stem's magnitude spectrogram before vs after 64 kbps AAC re-encode, showing
# the high-frequency band-limiting and quantization artefacts umxhq must adapt to.
#
#   import sys; sys.path.insert(0, '../../scripts'); import make_domains as md
#   cmds = md.aac64_commands(f'{SHARD_ROOT}/<track>')   # the pinned ffmpeg plan
#   for c in cmds: print(' '.join(c))                    # inspect (deterministic)
#   md.run_t1_aac64(SHARD_ROOT, execute=True)            # RUN LATER (needs ffmpeg)
#   # then STFT both stems and imshow log-magnitude side by side.
print('T1 spectrogram overlay — RUN LATER (needs a decoded shard + ffmpeg).')

## 6 · Merge round-trip demo

After training, `merge_lora` bakes $W_0+\frac{\alpha}{r}BA$ back into plain modules
(zero added inference cost): the merged model's forward equals the wrapped forward. The
cell wraps the mock host with LoRA, moves $B$ off zero (simulating a trained adapter),
and confirms the merged forward matches — for the Linears **and** the parametrized LSTM.

In [ ]:
# CPU-runnable later: full-model merge round-trip (wrapped forward == merged forward).
import torch
from torch import nn
from singnet.peft import load_umxhq, apply_recipe, merge_lora

model = load_umxhq('cpu', mock=True)
apply_recipe(model, 'lora16')
for m in model.modules():          # simulate a trained adapter: push every B off zero
    if hasattr(m, 'lora_B'):
        nn.init.normal_(m.lora_B, std=0.01)
model.eval()
x = torch.rand(2, 2, 2049, 6)
with torch.no_grad():
    before = model(x)
    merge_lora(model)              # bake in; removes LoRA params + LSTM parametrizations
    after = model(x)
print('merge round-trip — max abs diff:', float((before - after).abs().max()), '(≈0)')

## 7 · Takeaways

- The mock host is **shape-exact** to `umxhq` (8,893,348 params), so every accounting
  number here is the real one — the G1 recompute against the downloaded checkpoint is
  expected to reproduce it to the parameter.
- LoRA's $B=0$ init makes every LoRA run **start exactly at zero-shot** — a clean,
  bit-identical forgetting baseline (THEORY §6.2).
- `lora4`/`lora16` train **1.27%/4.85%** of the host; `head` trains **23.7%** — the
  efficiency gap the quality-vs-params curve (notebook 02) makes visible.
- The LSTM LoRA is the crux (THEORY §3); its $B=0$ identity and merge round-trip are
  bit-exact on CPU, de-risking the #1 engineering risk before any GPU spend.